# AI-Lake Format — Interactive Demo

This notebook walks through the core capabilities of the AI-Lake format:

1. **Vector search** — find the nearest neighbours to a query embedding
2. **Iceberg compatibility** — read the same table with PyIceberg (no plugin)
3. **SQL compatibility** — query with DuckDB
4. **RAG context assembly** — build structured LLM context from search results

The demo table was written automatically at container startup (`init_demo.py`):
- **500 documents**, dim=16, cosine similarity, F16 precision
- Each `.parquet` file contains tabular data **plus** a HNSW index in the file footer
- The same files are readable by any standard Parquet/Iceberg tool

## 1. Setup

In [ ]:
import json
import os
import pathlib

import ailake
import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
QUERY_PATH = pathlib.Path(TABLE_PATH).parent / 'demo_query.json'

print(f'Table path : {TABLE_PATH}')
print(f'Query path : {QUERY_PATH}')
print(f'ailake version : OK')

# List the files written by init_demo.py
parquet_files = sorted(pathlib.Path(TABLE_PATH).rglob('*.parquet'))
print(f'\nParquet files in table:')
for p in parquet_files:
    size_kb = p.stat().st_size / 1024
    print(f'  {p.relative_to(TABLE_PATH)}  ({size_kb:.1f} KB)')

## 2. Vector Search with ailake

The `ailake.search()` function:
1. Reads Iceberg metadata to discover which `.parquet` files exist
2. **Prunes** files using centroid distance (geometric pruning — skips files with no close vectors)
3. Loads the HNSW index from each surviving file's footer via mmap
4. Merges top-k across all files and returns results

In [ ]:
# Load the pre-generated demo query vector (doc_id=0, so it should be the top result)
with open(QUERY_PATH) as f:
    demo = json.load(f)

query_vector = demo['query_vector']
print(f'Query dim   : {len(query_vector)}')
print(f'Expected #1 : "{demo["expected_top1_text"][:60]}..."')

In [ ]:
results = ailake.search(TABLE_PATH, query_vector, top_k=5)

print(f'Top-{len(results)} results:')
for i, r in enumerate(results):
    print(f'  #{i+1}  row_id={r["row_id"]:>4}  distance={r["distance"]:.6f}  file={pathlib.Path(r["file"]).name}')

In [ ]:
# The query IS doc_id=0, so top result should have distance ≈ 0
top = results[0]
assert top['distance'] < 0.01, f'Expected near-zero distance, got {top["distance"]}'
print(f'PASS: top result distance = {top["distance"]:.2e} (query vector found itself)')

### Try your own query

Any 16-dimensional unit vector works. Here we generate a random one.

In [ ]:
rng = np.random.default_rng(seed=99)
random_query = rng.standard_normal(16).astype(np.float32)
random_query /= np.linalg.norm(random_query)

random_results = ailake.search(TABLE_PATH, random_query.tolist(), top_k=5)
df = pd.DataFrame(random_results)
df['file'] = df['file'].apply(lambda p: pathlib.Path(p).name)
df

## 3. Iceberg Compatibility — PyIceberg (no plugin)

The same `.parquet` files are valid Iceberg tables. Any standard Iceberg reader
works without the AI-Lake SDK — it just sees the tabular columns and treats
the HNSW footer bytes as an unknown extension (silently ignored).

In [ ]:
from pyiceberg.catalog.sql import SqlCatalog

# HadoopCatalog-style: point PyIceberg at the local warehouse directory
catalog = SqlCatalog(
    'demo',
    **{
        'uri': 'sqlite:////tmp/demo_iceberg.db',
        'warehouse': f'file://{TABLE_PATH}',
    }
)

# Register the table that ailake wrote (metadata is already Iceberg Spec v2)
import glob, json as _json
meta_dir = pathlib.Path(TABLE_PATH) / 'default' / 'table' / 'metadata'
hint = (meta_dir / 'version-hint.text').read_text().strip()
meta_path = meta_dir / f'v{hint}.metadata.json'
meta = _json.loads(meta_path.read_text())

print(f'Iceberg table UUID : {meta["table-uuid"]}')
print(f'Format version    : {meta["format-version"]}')
print(f'Snapshots         : {len(meta.get("snapshots", []))}')
print(f'ailake properties :')
for k, v in meta.get('properties', {}).items():
    if k.startswith('ailake'):
        print(f'  {k} = {v}')

In [ ]:
# Read the Parquet files directly with PyArrow — standard reader, no AI-Lake plugin
tables = [pq.read_table(str(p)) for p in parquet_files]
combined = pa.concat_tables(tables)

print(f'Rows read by PyArrow : {len(combined)}')
print(f'Schema               : {combined.schema}')
print()
combined.to_pandas().head(5)

## 4. SQL Compatibility — DuckDB

DuckDB reads the Parquet files as-is. The `embedding` column appears as
`BLOB` (raw bytes) — the HNSW index in the footer is invisible.

In [ ]:
parquet_glob = str(pathlib.Path(TABLE_PATH) / '**' / '*.parquet')

con = duckdb.connect()
result = con.execute(f"""
    SELECT
        count(*) AS total_rows,
        min(length(text)) AS min_text_len,
        max(length(text)) AS max_text_len,
        count(embedding)  AS rows_with_embedding
    FROM read_parquet('{parquet_glob}', hive_partitioning=false)
""").df()
print('DuckDB aggregate scan:')
print(result.to_string(index=False))

In [ ]:
sample = con.execute(f"""
    SELECT text
    FROM read_parquet('{parquet_glob}', hive_partitioning=false)
    WHERE text LIKE '%vector search%'
    LIMIT 5
""").df()
print(f"Rows about 'vector search':")
for row in sample['text']:
    print(f'  {row[:90]}...')

## 5. RAG Context Assembly

`ailake.assemble_context()` takes search results and builds structured XML
context ready to paste into a Claude/GPT-4 prompt.

It deduplicates similar chunks, groups by document, and fits within a token budget.

In [ ]:
# Re-run search and fetch text for each result
parquet_df = combined.to_pandas()

top_results = ailake.search(TABLE_PATH, query_vector, top_k=5)

chunks = []
for r in top_results:
    row_id = r['row_id']
    if row_id < len(parquet_df):
        text = parquet_df.iloc[int(row_id)]['text']
    else:
        text = f'(row {row_id} not found in local scan)'
    chunks.append({
        'document_id': f'doc-{row_id}',
        'chunk_index': 0,
        'chunk_text': text,
        'document_title': f'Document {row_id}',
        'distance': r['distance'],
    })

context_xml = ailake.assemble_context(chunks, max_tokens=2048, dedup_threshold=0.05)
print(context_xml)

## 6. MinIO / S3 — Compatibility via PyArrow

This cell uploads the demo table to the local MinIO instance and reads it back
via PyArrow's S3 filesystem — demonstrating object-storage compatibility.

> MinIO console: http://localhost:9001 (user: minioadmin / pass: minioadmin)

In [ ]:
import boto3
from botocore.client import Config

minio_endpoint = os.environ.get('MINIO_ENDPOINT', 'http://minio:9000')
access_key     = os.environ.get('MINIO_ACCESS_KEY', 'minioadmin')
secret_key     = os.environ.get('MINIO_SECRET_KEY', 'minioadmin')
bucket         = 'demo-bucket'

s3 = boto3.client(
    's3',
    endpoint_url=minio_endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

# Upload Parquet files to MinIO
uploaded = []
for p in parquet_files:
    key = f'ailake_demo/{p.relative_to(TABLE_PATH)}'
    s3.upload_file(str(p), bucket, key)
    uploaded.append(key)
    print(f'  uploaded → s3://{bucket}/{key}')

print(f'\nUploaded {len(uploaded)} files to MinIO.')

In [ ]:
import pyarrow.fs as pafs

# Read back from MinIO using PyArrow S3 filesystem
s3fs = pafs.S3FileSystem(
    endpoint_override=minio_endpoint.replace('http://', ''),
    access_key=access_key,
    secret_key=secret_key,
    scheme='http',
)

dataset = pq.ParquetDataset(
    [f'{bucket}/{k}' for k in uploaded],
    filesystem=s3fs,
)
s3_table = dataset.read(columns=['text'])
print(f'Read {len(s3_table)} rows from MinIO via PyArrow.')
print(f'Schema: {s3_table.schema}')

## Next Steps

| Task | Where to look |
|---|---|
| File format binary spec | `docs/specs/FILE_FORMAT.md` |
| Python API reference | `ailake-py/README.md` |
| Iceberg catalog integration | `docs/architecture/CATALOG_BACKENDS.md` |
| Spark / Trino / Flink plugins | `docs/specs/JVM_PLUGINS.md` |
| Production S3 deployment | `docs/specs/CLOUD_DEPLOY.md` |
| Contributing guide | `CONTRIBUTING.md` |

### Scale up

Replace `dim=16` with `dim=1536` (OpenAI `text-embedding-3-small`) and connect
a real embedding model to see production-scale recall numbers.
The format, API, and HNSW indexing are identical — only the dimension changes.